# KURE-v1 A_fixed Neo4j 적재

팀원이 전달한 KURE-v1 벡터를 STEP 7 A_fixed 청크에 연결해 적재합니다. Qdrant와 동일한 point_id, parent_id를 유지해 Qdrant 검색 결과에서 Neo4j 관계를 조회할 수 있습니다.

노트북 위치에서 위에서부터 실행합니다. 기존 정책·지침의 검수 상태는 그대로 보존합니다. 지역·대상 태그나 과거 v3 후보 링크를 자격 또는 승인된 관계로 승격하지 않습니다.

## 1. 전달 패키지와 원본 읽기

- chunk_vectors.jsonl에는 청크 ID와 적재할 벡터가 있습니다.
- chunk_vectors.npz는 같은 벡터의 NumPy 사본입니다.
- chunk_map.csv는 Qdrant UUID와 부모 ID, 출처를 확인하는 매핑표입니다.
- embedding_info.json은 모델, 차원, 버전, 원본과 패키지 해시를 기록합니다.
- 본문과 전체 메타데이터는 A_fixed의 정책·지침 청크에 있습니다.

In [1]:
# 패키지 읽기, 데이터 대조, Neo4j 연결에 필요한 모듈을 가져옵니다.
import csv
import hashlib
import json
import os
from collections import Counter
from pathlib import Path
from uuid import NAMESPACE_URL, uuid5

import numpy as np
from dotenv import load_dotenv
from IPython.display import display
from neo4j import GraphDatabase, RoutingControl

In [2]:
# 프로젝트 루트와 A_fixed 원본 및 KURE-v1 패키지 경로를 정합니다.
root = Path("../..").resolve()
package = root / "data/rag/embedding_by_KUREv1"
source_files = {
    kind: root / f"data/rag/chunks/A_fixed/{kind}_chunks.jsonl"
    for kind in ("policy", "guideline")
}
info = json.loads((package / "embedding_info.json").read_text(encoding="utf-8-sig"))
model = info["model"]
dimension = model["dim"]
display({key: info[key] for key in ("version", "chunk_set", "counts_by_type", "model")})

{'version': 'v1',
 'chunk_set': 'A_fixed',
 'counts_by_type': {'policy': 662, 'guideline': 2738},
 'model': {'name': 'nlpai-lab/KURE-v1',
  'revision': '8b418a58414668e75532ed045c22d9ca018ae2b2',
  'dim': 1024,
  'similarity': 'cosine',
  'normalize_embeddings': True,
  'dtype': 'float32',
  'preprocess': '줄바꿈을 공백으로 바꾼 뒤 임베딩 (LangChain HuggingFaceEmbeddings 기본 동작)',
  'input': '청크 text만 (제목 등을 앞에 붙이지 않음)',
  'instruction_prefix': None}}

In [3]:
# 전달받은 파일과 A_fixed 원본의 SHA-256을 명세와 대조합니다.
expected_hashes = {root / name: digest for name, digest in info["source_chunk_files"].items()}
expected_hashes.update({package / name: digest for name, digest in info["files"].items()})
hash_checks = {
    str(path.relative_to(root)): hashlib.sha256(path.read_bytes()).hexdigest() == digest
    for path, digest in expected_hashes.items()
}
display(hash_checks)
if not all(hash_checks.values()) or not set(source_files.values()) <= expected_hashes.keys():
    raise ValueError("패키지 또는 A_fixed 원본의 해시가 전달 명세와 다릅니다.")

{'data\\rag\\chunks\\A_fixed\\policy_chunks.jsonl': True,
 'data\\rag\\chunks\\A_fixed\\guideline_chunks.jsonl': True,
 'data\\rag\\embedding_by_KUREv1\\chunk_vectors.jsonl': True,
 'data\\rag\\embedding_by_KUREv1\\chunk_vectors.npz': True,
 'data\\rag\\embedding_by_KUREv1\\chunk_map.csv': True}

In [4]:
# 원본 청크, 벡터 JSONL, CSV 매핑표를 읽습니다.
def read_jsonl(path):
    with path.open(encoding="utf-8-sig") as stream:
        return [json.loads(line) for line in stream if line.strip()]

chunks_by_type = {kind: read_jsonl(path) for kind, path in source_files.items()}
vector_rows = read_jsonl(package / "chunk_vectors.jsonl")
with (package / "chunk_map.csv").open(encoding="utf-8-sig", newline="") as stream:
    map_rows = list(csv.DictReader(stream))

display({"원본 청크": {kind: len(rows) for kind, rows in chunks_by_type.items()},
         "JSONL 벡터": len(vector_rows), "CSV 매핑": len(map_rows)})

{'원본 청크': {'policy': 662, 'guideline': 2738}, 'JSONL 벡터': 3400, 'CSV 매핑': 3400}

## 2. ID와 벡터 전수 대조

chunk_id로 본문과 벡터를 연결합니다. point_id는 기존 UUID를 그대로 사용하며, 정책은 policy_id, 지침은 section_id를 부모 키로 씁니다. 잘못된 버전이나 서로 다른 청크가 들어오는 것을 막기 위해 적재 전에 전수 검사합니다.

In [5]:
# 청크·벡터·CSV의 건수와 키 집합, UUID, 부모 키, 출처를 모두 대조합니다.
source_rows = [row for rows in chunks_by_type.values() for row in rows]
chunks = {row["chunk_id"]: row for row in source_rows}
vectors = {row["chunk_id"]: row for row in vector_rows}
mapping = {row["chunk_id"]: row for row in map_rows}

shape_ok = (
    info["chunk_set"] == "A_fixed"
    and {kind: len(rows) for kind, rows in chunks_by_type.items()} == info["counts_by_type"]
    and len(source_rows) == len(vector_rows) == len(map_rows)
    == len(chunks) == len(vectors) == len(mapping) == info["n_chunks"]
    and chunks.keys() == vectors.keys() == mapping.keys()
)
if not shape_ok:
    raise ValueError("청크·벡터·CSV의 건수나 chunk_id가 일치하지 않습니다.")

id_errors = []
csv_fields = ("title", "sido_name", "sigungu_name", "region_scope", "source", "page_start", "page_end")
for kind, rows in chunks_by_type.items():
    parent_key = "policy_id" if kind == "policy" else "section_id"
    for row in rows:
        chunk_id = row["chunk_id"]
        meta, vector, entry = row["metadata"], vectors[chunk_id], mapping[chunk_id]
        if not (
            row["id"] == vector["point_id"] == entry["point_id"] == str(uuid5(NAMESPACE_URL, chunk_id))
            and row[parent_key] == meta["parent_id"] == vector["parent_id"] == entry["parent_id"]
            and kind == meta["record_type"] == vector["record_type"] == entry["record_type"]
            and chunk_id == meta["chunk_id"]
            and meta["chunk_strategy"] == "fixed"
            and all(entry[key] == ("" if meta.get(key) is None else str(meta[key])) for key in csv_fields)
        ):
            id_errors.append(chunk_id)

if id_errors or len({(row["record_type"], row["parent_id"]) for row in vector_rows}) != info["n_parents"]:
    raise ValueError(f"공통 ID 또는 부모·출처 정보가 일치하지 않습니다. {id_errors[:5]}")
print(f"공통 ID·출처 대조 PASS ({len(chunks):,}개)")

공통 ID·출처 대조 PASS (3,400개)


In [6]:
# JSONL 벡터의 차원·유한성·정규화와 NPZ의 float32 값 일치를 전수 확인합니다.
with np.load(package / "chunk_vectors.npz", allow_pickle=False) as archive:
    npz_ids = archive["chunk_ids"].tolist()
    npz_vectors = archive["vectors"]

matrix = np.asarray([vectors[chunk_id]["vector"] for chunk_id in npz_ids], dtype=np.float64)
vector_checks = {
    "모델과 설정": model["name"] == "nlpai-lab/KURE-v1" and model["normalize_embeddings"]
                  and model["similarity"] == "cosine" and dimension == 1024,
    "NPZ 청크 ID": len(npz_ids) == len(vectors) and set(npz_ids) == vectors.keys(),
    "벡터 차원과 자료형": matrix.shape == npz_vectors.shape == (len(vectors), dimension)
                        and str(npz_vectors.dtype) == model["dtype"],
    "유한 벡터": bool(np.isfinite(matrix).all()),
    "L2 정규화": bool(np.allclose(np.linalg.norm(matrix, axis=1), 1, atol=1e-5, rtol=0)),
    "JSONL과 NPZ 값 일치": bool(np.array_equal(matrix.astype(npz_vectors.dtype), npz_vectors)),
}
display(vector_checks)
if not all(vector_checks.values()):
    raise ValueError("전달받은 벡터의 설정이나 값이 일치하지 않습니다.")

{'모델과 설정': True,
 'NPZ 청크 ID': True,
 '벡터 차원과 자료형': True,
 '유한 벡터': True,
 'L2 정규화': True,
 'JSONL과 NPZ 값 일치': True}

## 3. Neo4j 속성과 관계 준비

청크는 원본 chunk_id로 식별하고 Qdrant point_id를 함께 저장합니다. 정책·지침 청크는 CHUNK_OF로 각 부모에 연결합니다. 원본 PDF는 HAS_SECTION으로 실제 섹션에 연결합니다. 대상 태그, 미검증 지역, PDF 페이지 범위, 이미지 참조 등은 청크의 원본 metadata_json에도 보존합니다.

In [7]:
# 원본의 청크 속성과 실제 부모·PDF ID를 적재용으로 구성합니다.
prepared = {}
for kind, rows in chunks_by_type.items():
    parent_key = "policy_id" if kind == "policy" else "section_id"
    prepared[kind] = []
    for row in rows:
        meta = row["metadata"]
        chunk_id = row["chunk_id"]
        parent_id = row[parent_key]
        vector = vectors[chunk_id]
        parent = {key: value for key, value in meta.items()
                  if key not in {"images", "chunk_id", "chunk_strategy", "parent_id"}}
        parent[parent_key] = parent_id
        chunk = {key: value for key, value in meta.items() if key != "images"}
        chunk.update({
            parent_key: parent_id,
            "id": row["id"],
            "point_id": vector["point_id"],
            "text": row["text"],
            "strategy": info["chunk_set"],
            "embedding": vector["vector"],
            "embedding_model": model["name"],
            "embedding_revision": model["revision"],
            "embedding_dim": dimension,
            "embedding_package_version": info["version"],
            "embedding_package_sha256": info["files"]["chunk_vectors.jsonl"],
            "metadata_json": json.dumps(meta, ensure_ascii=False, sort_keys=True),
        })
        item = {"chunk": chunk, "parent": parent}
        if kind == "guideline":
            item["document"] = {
                key: meta[key] for key in ("document_id", "source", "title", "source_type",
                                           "publisher", "publication_year", "source_url")
            }
        prepared[kind].append(item)

display({kind: {"청크": len(rows), "부모": len({row["chunk"]["parent_id"] for row in rows})}
         for kind, rows in prepared.items()})

{'policy': {'청크': 662, '부모': 574}, 'guideline': {'청크': 2738, '부모': 1189}}

In [8]:
# 프로젝트 .env에서 Neo4j 연결 정보를 읽고 대상 데이터베이스에 연결합니다.
load_dotenv(root / ".env")
database = os.getenv("NEO4J_DATABASE") or "neo4j"
driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI") or "bolt://localhost:7687",
    auth=(os.getenv("NEO4J_USER") or os.getenv("NEO4J_ID") or "neo4j",
          os.getenv("NEO4J_PASSWORD") or os.environ["NEO4J_PW"]),
)
driver.verify_connectivity()
records = driver.execute_query("RETURN 1 AS connected", database_=database,
                               routing_=RoutingControl.READ).records
display({"데이터베이스": database, "연결": records[0]["connected"]})

{'데이터베이스': 'neo4j', '연결': 1}

In [9]:
# 청크의 논리 ID와 UUID, 부모 노드 ID에 유일성 제약조건을 적용합니다.
constraints = (
    ("RagABChunk", "chunk_id"),
    ("RagABChunk", "id"),
    ("RagABChunk", "point_id"),
    ("RagABPolicy", "policy_id"),
    ("RagABSection", "section_id"),
    ("RagABDocument", "document_id"),
)
for label, field in constraints:
    driver.execute_query(
        f"CREATE CONSTRAINT rag_ab_{label.lower()}_{field} IF NOT EXISTS "
        f"FOR (n:{label}) REQUIRE n.{field} IS UNIQUE",
        database_=database,
    )

## 4. A_fixed 배치 적재

MERGE로 같은 ID를 다시 적재해도 청크와 관계를 재사용합니다. 저장된 벡터는 전달받은 JSONL의 값입니다.

In [10]:
# 청크·부모·원본 문서를 원본 ID로 연결하는 Cypher를 정의합니다.
merge_chunk = '''
UNWIND $rows AS row
MERGE (c:RagABChunk {chunk_id: row.chunk.chunk_id})
SET c:RagABFixedChunk
SET c += row.chunk
'''
parent_queries = {
    "policy": '''
MERGE (p:RagABPolicy {policy_id: row.parent.policy_id})
SET p += row.parent
MERGE (c)-[:CHUNK_OF]->(p)
RETURN count(c) AS written
''',
    "guideline": '''
MERGE (s:RagABSection {section_id: row.parent.section_id})
SET s += row.parent
MERGE (d:RagABDocument {document_id: row.document.document_id})
SET d += row.document
MERGE (c)-[:CHUNK_OF]->(s)
MERGE (d)-[:HAS_SECTION]->(s)
RETURN count(c) AS written
''',
}

In [11]:
# 정책과 지침 청크를 각각 200개씩 적재합니다.
written = Counter()
for kind, rows in prepared.items():
    for start in range(0, len(rows), 200):
        records = driver.execute_query(
            merge_chunk + parent_queries[kind], rows=rows[start:start + 200], database_=database,
        ).records
        written[kind] += records[0]["written"]

display(dict(written))

{'policy': 662, 'guideline': 2738}

## 5. 저장된 값과 그래프 경로 검수

Neo4j에서 다시 읽은 청크의 속성과 벡터를 원본 및 전달 패키지와 전수 대조합니다. 정책·섹션·원본 PDF 경로도 청크별로 확인합니다.

In [12]:
# Neo4j에서 청크, 벡터, 부모와 PDF 경로 수를 집계합니다.
summary_query = '''
MATCH (c:RagABFixedChunk)
RETURN c.record_type AS kind, count(*) AS chunks, count(c.embedding) AS vectors,
       count(DISTINCT c.parent_id) AS parents
ORDER BY kind
'''
summary = driver.execute_query(summary_query, database_=database, routing_=RoutingControl.READ).records
display([row.data() for row in summary])

[{'kind': 'guideline', 'chunks': 2738, 'vectors': 2738, 'parents': 1189},
 {'kind': 'policy', 'chunks': 662, 'vectors': 662, 'parents': 574}]

In [13]:
# 입력과 Neo4j를 청크 ID 단위로 대조하고 본문·벡터·메타데이터·그래프를 확인합니다.
audit_query = '''
UNWIND $chunk_ids AS chunk_id
MATCH (c:RagABChunk:RagABFixedChunk {chunk_id: chunk_id})
RETURN properties(c) AS chunk,
       [(c)-[:CHUNK_OF]->(p) | coalesce(p.policy_id, p.section_id)] AS parents,
       [(c)-[:CHUNK_OF]->(:RagABSection)<-[:HAS_SECTION]-(d:RagABDocument) | d.document_id] AS documents
'''
audit_errors = []
audited = 0
for kind, rows in prepared.items():
    for start in range(0, len(rows), 200):
        batch = {row["chunk"]["chunk_id"]: row for row in rows[start:start + 200]}
        records = driver.execute_query(audit_query, chunk_ids=list(batch), database_=database,
                                       routing_=RoutingControl.READ).records
        for result in records:
            actual = result["chunk"]
            chunk_id = actual["chunk_id"]
            expected = {key: value for key, value in batch[chunk_id]["chunk"].items() if value is not None}
            expected_documents = [expected["document_id"]] if kind == "guideline" else []
            if (actual != expected or result["parents"] != [expected["parent_id"]]
                    or result["documents"] != expected_documents):
                audit_errors.append(chunk_id)
        audited += len(records)

counts_ok = {row["kind"]: row["chunks"] for row in summary} == info["counts_by_type"]
vectors_ok = {row["kind"]: row["vectors"] for row in summary} == info["counts_by_type"]
if (not counts_ok or not vectors_ok or written != Counter(info["counts_by_type"])
        or audited != len(chunks) or audit_errors):
    raise ValueError(f"적재 검수 불일치. 조회 {audited:,}개, 불일치 ID {audit_errors[:5]}")
print(f"Neo4j 본문·메타데이터·벡터·관계 전수 대조 PASS ({audited:,}개)")

Neo4j 본문·메타데이터·벡터·관계 전수 대조 PASS (3,400개)


## 6. 조회 테스트

서비스의 Graph 비교는 같은 Qdrant 검색 결과를 사용합니다. Graph를 추가한 쪽에서만 parent_id로 Neo4j 관계를 조회합니다. 아래는 Qdrant에서 쓰는 ID를 그대로 넣어 경로를 확인하는 예시입니다.

Neo4j 벡터 인덱스는 전달받은 청크 벡터로만 동작을 점검합니다. 이 테스트는 지역 필터를 적용한 서비스 검색이나 Graph 사용·미사용 성능 비교와 별개입니다.

In [14]:
# Qdrant와 같은 UUID·부모 ID로 정책과 PDF 출처를 조회합니다.
graph_lookup = '''
MATCH (c:RagABChunk:RagABFixedChunk {point_id: $point_id})-[:CHUNK_OF]->(p)
WHERE (p:RagABPolicy AND p.policy_id = $parent_id)
   OR (p:RagABSection AND p.section_id = $parent_id)
OPTIONAL MATCH (d:RagABDocument)-[:HAS_SECTION]->(p)
RETURN c.point_id AS point_id, c.parent_id AS parent_id, c.record_type AS record_type,
       p.title AS title, p.source_url AS source_url, d.document_id AS document_id,
       d.source AS source, p.page_start AS page_start, p.page_end AS page_end,
       p.region_selection_verified AS region_selection_verified
'''
graph_examples = []
for rows in prepared.values():
    probe = rows[0]["chunk"]
    result = driver.execute_query(
        graph_lookup, point_id=probe["point_id"], parent_id=probe["parent_id"],
        database_=database, routing_=RoutingControl.READ,
    ).records
    graph_examples.append(result[0].data())

display(graph_examples)

[{'point_id': 'ca897d6a-30c1-5db0-8b46-dedcaeab5c54',
  'parent_id': 'local:WLF00006795',
  'record_type': 'policy',
  'title': '서울형 장애인 개인예산제 시범사업',
  'source_url': 'https://www.bokjiro.go.kr/ssis-tbu/twataa/wlfareInfo/moveTWAT52011M.do?wlfareInfoId=WLF00006795&wlfareInfoReldBztpCd=02',
  'document_id': None,
  'source': None,
  'page_start': None,
  'page_end': None,
  'region_selection_verified': False},
 {'point_id': '6bb16a05-8049-5ced-b921-003f6c7490b1',
  'parent_id': 'welfare_services_2026_p0003_01',
  'record_type': 'guideline',
  'title': '2026 나에게 힘이 되는 복지서비스',
  'source_url': None,
  'document_id': 'mohw_welfare_services_2026',
  'source': '2026_나에게_힘이_되는_복지서비스',
  'page_start': 3,
  'page_end': 3,
  'region_selection_verified': None}]

In [15]:
# A_fixed 전용 코사인 벡터 인덱스를 생성하고 ONLINE 상태를 확인합니다.
index_name = "rag_a_fixed_kure_v1_embedding"
driver.execute_query(
    f"CREATE VECTOR INDEX {index_name} IF NOT EXISTS "
    "FOR (c:RagABFixedChunk) ON (c.embedding) "
    "OPTIONS {indexConfig: {`vector.dimensions`: $dimension, `vector.similarity_function`: $similarity}}",
    dimension=dimension, similarity=model["similarity"], database_=database,
)
driver.execute_query("CALL db.awaitIndex($name, 300)", name=index_name, database_=database)
indexes = driver.execute_query(
    "SHOW VECTOR INDEXES YIELD name, state, labelsOrTypes, options "
    "WHERE name = $name RETURN name, state, labelsOrTypes, options.indexConfig AS config",
    name=index_name, database_=database, routing_=RoutingControl.READ,
).records
index = indexes[0].data()
display(index)
if (index["state"] != "ONLINE" or index["labelsOrTypes"] != ["RagABFixedChunk"]
        or index["config"]["vector.dimensions"] != dimension
        or index["config"]["vector.similarity_function"].casefold() != model["similarity"].casefold()):
    raise ValueError("A_fixed 벡터 인덱스의 설정 또는 상태가 예상과 다릅니다.")

{'name': 'rag_a_fixed_kure_v1_embedding',
 'state': 'ONLINE',
 'labelsOrTypes': ['RagABFixedChunk'],
 'config': {'vector.dimensions': 1024,
  'vector.default_search_expansion_factor': 3.0,
  'vector.hnsw.m': 16,
  'vector.quantization.type': 'BINARY',
  'vector.similarity_function': 'COSINE',
  'vector.hnsw.ef_construction': 100}}

In [16]:
# 원본 정책·지침 벡터로 상위 5개를 조회해 벡터 인덱스 작동을 확인합니다.
vector_search = '''
CALL db.index.vector.queryNodes($name, 5, $vector)
YIELD node, score
RETURN node.chunk_id AS chunk_id, node.point_id AS point_id,
       node.parent_id AS parent_id, node.record_type AS record_type, score
ORDER BY score DESC
'''
for kind, rows in prepared.items():
    probe = rows[0]["chunk"]
    records = driver.execute_query(vector_search, name=index_name, vector=probe["embedding"],
                                   database_=database, routing_=RoutingControl.READ).records
    display({"입력": probe["chunk_id"], "유형": kind, "검색 결과": [r.data() for r in records]})
    if len(records) != 5 or not any(row["chunk_id"] == probe["chunk_id"] for row in records):
        raise ValueError(f"벡터 검색에서 입력 청크를 확인하지 못했습니다. {probe['chunk_id']}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nCALL db.index.vector.queryNodes($name, 5, $vector)\nYIELD node, score\nRETURN node.chunk_id AS chunk_id, node.point_id AS point_id,\n       node.parent_id AS parent_id, node.record_type AS record_type, score\nORDER BY score DESC\n'


{'입력': 'policy::local:WLF00006795::fixed::0001',
 '유형': 'policy',
 '검색 결과': [{'chunk_id': 'policy::local:WLF00006795::fixed::0001',
   'point_id': 'ca897d6a-30c1-5db0-8b46-dedcaeab5c54',
   'parent_id': 'local:WLF00006795',
   'record_type': 'policy',
   'score': 1.0},
  {'chunk_id': 'policy::local:WLF00002754::fixed::0001',
   'point_id': 'a71d0063-65fc-5f56-b2c1-de5d69b69d8b',
   'parent_id': 'local:WLF00002754',
   'record_type': 'policy',
   'score': 0.8802129626274109},
  {'chunk_id': 'policy::local:WLF00002796::fixed::0001',
   'point_id': '96a6f15e-c68e-51f3-8d66-b47eeb019d18',
   'parent_id': 'local:WLF00002796',
   'record_type': 'policy',
   'score': 0.8718785047531128},
  {'chunk_id': 'policy::local:WLF00000846::fixed::0001',
   'point_id': '8b2ca788-68d5-5573-99aa-2eb898c889ac',
   'parent_id': 'local:WLF00000846',
   'record_type': 'policy',
   'score': 0.8715559244155884},
  {'chunk_id': 'policy::local:WLF00005409::fixed::0001',
   'point_id': '557760e0-0c7c-56af-8697-e03

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nCALL db.index.vector.queryNodes($name, 5, $vector)\nYIELD node, score\nRETURN node.chunk_id AS chunk_id, node.point_id AS point_id,\n       node.parent_id AS parent_id, node.record_type AS record_type, score\nORDER BY score DESC\n'


{'입력': 'guideline::welfare_services_2026_p0003_01::fixed::0001',
 '유형': 'guideline',
 '검색 결과': [{'chunk_id': 'guideline::welfare_services_2026_p0003_01::fixed::0001',
   'point_id': '6bb16a05-8049-5ced-b921-003f6c7490b1',
   'parent_id': 'welfare_services_2026_p0003_01',
   'record_type': 'guideline',
   'score': 1.0},
  {'chunk_id': 'guideline::basic_pension_2026_p0315_01::fixed::0001',
   'point_id': '44b53f1f-8caa-5ec8-9582-7b31ca858164',
   'parent_id': 'basic_pension_2026_p0315_01',
   'record_type': 'guideline',
   'score': 0.8474057912826538},
  {'chunk_id': 'guideline::welfare_services_2026_p0010_01::fixed::0001',
   'point_id': 'd2036387-bbb1-547e-be19-391c8a14f1d2',
   'parent_id': 'welfare_services_2026_p0010_01',
   'record_type': 'guideline',
   'score': 0.8358075618743896},
  {'chunk_id': 'guideline::welfare_services_2026_p0382_01::fixed::0001',
   'point_id': 'aa0e62bc-c1e3-5b39-ac1e-ac3c28b62d52',
   'parent_id': 'welfare_services_2026_p0382_01',
   'record_type': 'guid

In [17]:
# 검수를 마친 뒤 Neo4j 연결을 닫습니다.
driver.close()

## 7. Neo4j 데이터베이스 내보내기

백업이 필요한 때 아래 셀을 순서대로 실행합니다. 중간 셀은 Neo4j 서비스에 잠깐 연결할 수 없게 하므로 공유 중이라면 팀에 먼저 알립니다. 현재 컨테이너의 실제 /data 마운트와 이미지 ID를 사용해, 이전 Compose 위치에서 만들어진 컨테이너도 올바르게 백업합니다.

덤프에는 지정한 DB의 노드·관계·속성·벡터와 스키마가 들어갑니다. 사용자·역할 정보가 있는 system DB, .env, 외부 PDF·이미지 파일은 별도로 관리합니다. 팀에 전달할 때는 출력에 표시된 Neo4j 버전도 함께 기록합니다. [Neo4j Docker 덤프·복원](https://neo4j.com/docs/operations-manual/current/docker/dump-load/) · [오프라인 백업 범위](https://neo4j.com/docs/operations-manual/current/backup-restore/offline-backup/)

### 7.1 백업 대상과 저장 경로 확인

루트 Compose에서 서비스 이름을 확인하고 Docker에서 현재 컨테이너의 실제 마운트와 이미지 버전을 읽습니다.

In [ ]:
# 현재 Neo4j 컨테이너의 데이터 경로와 백업 폴더를 확인합니다.
import json
import os
import subprocess
from datetime import datetime
from pathlib import Path

from dotenv import dotenv_values

backup_root = Path("../..").resolve()
compose_file = backup_root / "docker-compose.yml"
env_file = backup_root / ".env"
compose = ["docker", "compose", "--env-file", str(env_file), "-f", str(compose_file)]
config = json.loads(subprocess.check_output([*compose, "config", "--format", "json"]))
service = config["services"]["neo4j"]
backup_database = os.getenv("NEO4J_DATABASE") or dotenv_values(env_file).get("NEO4J_DATABASE") or "neo4j"
if backup_database.casefold() == "system":
    raise ValueError("팀 공유용 덤프에는 사용자 데이터베이스를 지정하세요.")

container = json.loads(subprocess.check_output(["docker", "inspect", service["container_name"]]))[0]
if not container["State"]["Running"] or container["Config"]["Labels"].get("com.docker.compose.service") != "neo4j":
    raise RuntimeError("실행 중인 Neo4j Compose 서비스 컨테이너를 확인하세요.")
container_id = container["Id"]
data_mount = next(mount for mount in container["Mounts"] if mount["Destination"] == "/data")
if data_mount["Type"] != "bind":
    raise RuntimeError("현재 프로젝트의 Neo4j /data는 호스트 폴더에 바인드되어 있어야 합니다.")

admin = ["docker", "run", "--rm", "--pull", "never", "--network", "none", "--hostname", "localhost",
         "--user", "neo4j", "--entrypoint", "neo4j-admin"]
image_id = container["Image"]
backup_version = subprocess.check_output([*admin, image_id, "--version"], text=True).strip()
backup_dir = Path.cwd() / "database" / "backups" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
dump_file = backup_dir / f"{backup_database}.dump"
print(f"Neo4j {backup_version} | DB: {backup_database}")
print(f"데이터 폴더: {data_mount['Source']}")
print(f"덤프 파일: {dump_file}")

### 7.2 Neo4j 중지, 덤프, 재시작

한 셀에서 중지부터 재시작까지 처리합니다. 오류가 발생해도 같은 컨테이너를 다시 시작하도록 시도합니다.

In [ ]:
# 같은 Neo4j 이미지의 관리 명령으로 오프라인 덤프를 만듭니다.
backup_dir.mkdir(parents=True, exist_ok=False)
try:
    subprocess.run(["docker", "stop", "--time", "120", container_id], check=True)
    subprocess.run(
        [*admin,
         "--mount", f"type=bind,source={data_mount['Source']},target=/data",
         "--mount", f"type=bind,source={backup_dir},target=/backups",
         image_id, "database", "dump", backup_database, "--to-path=/backups"],
        check=True,
    )
finally:
    subprocess.run(["docker", "start", container_id], check=True)

### 7.3 생성된 덤프 확인

백업 파일의 경로와 크기를 표시합니다. 팀원에게는 이 파일과 Neo4j 버전을 전달합니다.

In [ ]:
# 덤프가 존재하며 비어 있지 않은지 확인합니다.
dump_size = dump_file.stat().st_size
if not dump_size:
    raise RuntimeError(f"덤프 파일이 비어 있습니다. {dump_file}")
print(f"{dump_file} ({dump_size:,} bytes)")

## 8. 팀원용 수동 복원

PowerShell을 notebooks/Neo4j_test 폴더에서 열고, 전달받은 neo4j.dump를 이 폴더의 database/backups/ 바로 아래에 넣습니다. 덤프를 만든 Neo4j와 같은 버전의 컨테이너를 사용합니다. 이번 적재 환경은 Neo4j 2026.10.2입니다. 기존 데이터가 있다면 새 데이터 폴더를 사용하는 것을 권장합니다. [Neo4j 복원 공식 문서](https://neo4j.com/docs/operations-manual/current/backup-restore/restore-dump/)

### 8.1 복원할 컨테이너와 경로 확인

새 팀원에게 neo4j 컨테이너가 아직 없으면, 루트 Compose의 neo4j 이미지 태그를 덤프와 같은 버전으로 지정하고 아래 첫 블록을 실행해 컨테이너를 준비합니다. 현재 루트 Compose에서 새로 만든 컨테이너는 프로젝트 루트의 database/neo4j/data를 /data로 사용합니다.

```powershell
$projectRoot = (Resolve-Path ..\..).Path
docker compose --env-file "$projectRoot\.env" -f "$projectRoot\docker-compose.yml" create neo4j
```

다음 블록에서 출력한 데이터 폴더가 실제 복원 대상인지 확인합니다. 이전 Compose 위치에서 만든 컨테이너는 notebooks/Neo4j_test/database/neo4j_data를 사용할 수 있습니다. 프로젝트 .env에서 NEO4J_DATABASE를 지정했다면 $db를 그 이름으로 바꾸고 파일 이름도 확인합니다.

```powershell
$db = 'neo4j'
$backupDir = (Resolve-Path ./database/backups -ErrorAction Stop).Path
if (-not (Test-Path -LiteralPath (Join-Path $backupDir "$db.dump") -PathType Leaf)) {
    throw "$db.dump 파일을 database/backups에 넣으세요."
}

$rawContainer = docker inspect neo4j
if ($LASTEXITCODE -ne 0) { throw 'Neo4j 컨테이너를 확인할 수 없습니다.' }
$target = @($rawContainer -join [Environment]::NewLine | ConvertFrom-Json)[0]
$data = $target.Mounts | Where-Object { $_.Destination -eq '/data' }
if ($target.Config.Labels.'com.docker.compose.service' -ne 'neo4j' -or $data.Type -ne 'bind') {
    throw 'Neo4j Compose 서비스의 /data 마운트를 확인하세요.'
}
$admin = @('run', '--rm', '--pull', 'never', '--network', 'none', '--hostname', 'localhost', '--user', 'neo4j', '--entrypoint', 'neo4j-admin')
docker @admin $target.Image --version
if ($LASTEXITCODE -ne 0) { throw '대상 Neo4j 버전을 확인할 수 없습니다.' }
"복원 대상 데이터 폴더 = $($data.Source)"
```

### 8.2 중지 → 복원 → 재시작

**기존 DB가 있으면 기본 설정에서 복원이 거부됩니다.** 기존 데이터를 백업하고 덮어쓸 대상을 확인한 뒤에만 $overwrite 값을 true로 바꿉니다. 이 명령은 Neo4j Browser의 Cypher 창이 아닌 PowerShell에서 직접 실행합니다.

```powershell
$overwrite = 'false'
try {
    if ($target.State.Running) {
        docker stop --time 120 $target.Id
        if ($LASTEXITCODE -ne 0) { throw 'Neo4j를 중지하지 못했습니다.' }
    }

    docker @admin --mount "type=bind,source=$($data.Source),target=/data" --mount "type=bind,source=$backupDir,target=/backups,readonly" $target.Image database load $db --from-path=/backups "--overwrite-destination=$overwrite"
    if ($LASTEXITCODE -ne 0) { throw 'Neo4j 데이터베이스 복원에 실패했습니다.' }
} finally {
    docker start $target.Id
    if ($LASTEXITCODE -ne 0) { throw 'Neo4j를 시작하지 못했습니다.' }
}
```

### 8.3 복원 결과 확인

Neo4j Browser에서 대상 DB에 접속해 각각 실행합니다. 현재 A_fixed 기준으로 정책 662개와 지침 2,738개에 각각 1,024차원 벡터가 있고, CHUNK_OF 3,400개와 HAS_SECTION 1,189개가 있어야 합니다. 벡터 인덱스가 ONLINE인지 확인합니다.

```cypher
MATCH (c:RagABFixedChunk)
RETURN c.record_type AS kind, count(c) AS chunks, count(c.embedding) AS vectors,
       min(size(c.embedding)) AS min_dim, max(size(c.embedding)) AS max_dim;
```

```cypher
MATCH (c:RagABFixedChunk)-[r:CHUNK_OF]->(parent)
OPTIONAL MATCH (:RagABDocument)-[s:HAS_SECTION]->(parent)
RETURN count(DISTINCT r) AS chunk_of, count(DISTINCT s) AS has_section;
```

```cypher
SHOW VECTOR INDEXES YIELD name, state
WHERE name = 'rag_a_fixed_kure_v1_embedding'
RETURN name, state;
```